<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/FinalWork/huggingknn_final3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

 # Zero-shot classification using a pretrained NLI model

In [ ]:
from transformers import pipeline

# Model:"joeddav/xlm-roberta-large-xnli"
# This model takes xlm-roberta-large and fine-tunes it on a combination of NLI data in 15 languages.
# This model is intended to be used for zero-shot text classification, especially in languages other than English. It is fine-tuned on XNLI, which is a multilingual NLI dataset.

pipe = pipeline("zero-shot-classification",model="joeddav/xlm-roberta-large-xnli")


We load a zero-shot classification pipeline from Hugging Face.

The model joeddav/xlm-roberta-large-xnli is:

  * Based on XLM-RoBERTa-large
  * Fine-tuned on XNLI (Cross-lingual Natural Language Inference) data
  * Designed to classify text without task-specific training

Zero-shot classification works by reframing classification as an NLI task:
  * Premise: the input text (job position)
  * Hypothesis: “This text belongs to category X”

This makes the model suitable when labeled training data is limited or unavailable.

In [ ]:
df = pd.read_csv("/content/DataScienceCapstoneProject/department.csv")
# Predict department with a pretrained model based on position name
dpt_labels = df["label"].drop_duplicates()
text = df["text"].tolist()
candidate_labels = dpt_labels
result = pipe(text, candidate_labels)

* candidate_labels are all unique department names in the dataset.
* Each job position is classified against all department labels.
* The model returns a ranked list of labels for each position with confidence scores.

The dataset contains job profiles with fields such as:
position, department, seniority

# Zero-shot prediction of department (Department.csv)

In [ ]:
df = pd.read_csv("/content/DataScienceCapstoneProject/department.csv")

In [ ]:
prediction = []
# For each position, we take the top-ranked predicted department.
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

# Evaluation
from sklearn.metrics import classification_report
true_label = df["label"].astype(str)
print(classification_report(true_label, prediction))

# Result interpretation (Department.csv file)

## Accuracy = 74%
When we use only the job title to predict the department in the "department.csv" file, xlm-roberta-large-xnli captures semantic meaning of job titles.
Especially useful in multilingual or non-standard job naming contexts.

#

# Zero-shot prediction of department (Linkedin CV test data)

In [ ]:
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
# Predict department with a pretrained model based on position name
dpt_labels = df["department"].drop_duplicates()
text = df["position"].tolist()
candidate_labels = dpt_labels
result = pipe(text, candidate_labels)

prediction = []
# For each position, we take the top-ranked predicted department.
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

# Evaluation
from sklearn.metrics import classification_report
true_label = df["department"].astype(str)
print(classification_report(true_label, prediction))


In [ ]:
# Department values counts from "department.csv"
departments = pd.read_csv("/content/DataScienceCapstoneProject/department.csv")
counts1 = departments["label"].value_counts()
# Department values counts from linkedin CV
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
counts2 = df["department"].value_counts()

counts = pd.DataFrame(
    {
        "TrainDataset": counts1,
        "TestDataset": counts2
    }
)
counts.plot(kind = "bar", figsize = (10,6))
plt.xlabel("Departments")
plt.ylabel("Counts")
plt.title("Department Counts Comparison")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Result interpretation (Linkedin CV test file)
Accuracy = 56%

When we use only the job title to predict the department in the linkedin CV file, xlm-roberta-large-xnli reached a lower accuracy.

One of the possible reasons is that the label distribution here in the CV test data is quite different from the Department dataset.
For example, in the Department dataset,it includes around
* 40% is Marketing,
* 30% is Sales,
* 10% Information technology,
* only 0.42% is Other.

But in the linkedin CV test dataset,
* almost 50% of the data is Other department.

This Label distribution shift leads to the big difference in prediction accuracy.

Additionally, using job titles alone provides limited semantic information, further increasing ambiguity.

As a result, accuracy is not a reliable metric under these conditions, and the observed performance degradation is expected.

### Let's test the accuracy on the Linkedin CV file without "Other" department


In [ ]:
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
df = df[df["department"]!= "Other"]
# Predict department with a pretrained model based on position name
dpt_labels = df["department"].drop_duplicates().astype(str).tolist()
text = df["position"].tolist()
candidate_labels = dpt_labels
result = pipe(text, candidate_labels)

prediction = []
# For each position, we take the top-ranked predicted department.
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

# Evaluation
from sklearn.metrics import classification_report
true_label = df["department"].astype(str)
print(classification_report(true_label, prediction))

After removing the data labeled with "Other" department, we see an  increase in the general accuracy of the zero shot prediction, leading to 64%. But compared to the 56% accuracy before the "Other" department data is removed, this is not a huge increase.

# Zero-shot prediction of seniority (seniority.csv)

In [ ]:
seniority = pd.read_csv("/content/DataScienceCapstoneProject/seniority.csv")
# Predict seniority with a pretrained model based on position name
sen_labels = seniority["label"].drop_duplicates()
text = seniority["text"].tolist()
candidate_labels = sen_labels
result = pipe(text, candidate_labels)

prediction = []
# For each position, we take the top-ranked predicted seniority.
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

# Evaluation
from sklearn.metrics import classification_report
true_label = seniority["label"].astype(str)
print(classification_report(true_label, prediction))

# Zero-shot prediction of seniority  (Linkedin CV test file)


In [ ]:
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
sen_labels = df["seniority"].drop_duplicates()
text = df["position"].tolist()
candidate_labels = sen_labels
result = pipe(text, candidate_labels)

# The same zero-shot approach is applied.
# Only the label space changes from departments → seniority levels.

prediction = []
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

from sklearn.metrics import classification_report
print(classification_report(df["seniority"], prediction))

# Result interpretation (Seniority)
## Accuracy = 52%

Seniority terms (e.g., Junior, Lead, Manager) are:
* More subtle
* Often ambiguous in job titles

Zero-shot models struggle when label distinctions are not explicit in text.

## Zero shot prediction on seniority without "Professional" labeled data (Linkedin CV test file)

In [ ]:
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
df = df[df["seniority"] != "Professional"]
sen_labels = df["seniority"].drop_duplicates()
text = df["position"].tolist()
candidate_labels = sen_labels
result = pipe(text, candidate_labels)

# The same zero-shot approach is applied.
# Only the label space changes from departments → seniority levels.

prediction = []
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

from sklearn.metrics import classification_report
print(classification_report(df["seniority"], prediction))

# Attempt to fine-tune the model on labeled department data

xlm-roberta-large-xnli showed acceptable overall zero-shot performance, and we attempted to:

* Fine-tune it using department dataset
* Evaluate on Linkedin CV dataset
* Expect improved task-specific accuracy

### Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Departments are converted from strings → numeric labels
# Required for supervised training
departments = pd.read_csv("/content/DataScienceCapstoneProject/department.csv")
le = LabelEncoder()
codes = le.fit_transform(departments["label"].drop_duplicates())
dpt = le.inverse_transform(codes)
mapping = dict(zip(dpt, codes))

departments["labels"] = departments["label"].map(mapping)
train_dataset = departments.drop(columns = "label").rename(columns={"text":"position"})

# Encode "department" in evaluation dataset
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")[["position","department"]]
df["labels"] = df["department"].map(mapping)
eval_dataset = df.drop(columns = "department")

In [ ]:
eval_dataset

### Prepare training and evaluation dataset



In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_dataset)
eval_dataset = Dataset.from_pandas(eval_dataset)

In [ ]:
eval_dataset

In [ ]:
train_dataset

### Tokenization

In [ ]:
from transformers import AutoTokenizer

model_name = "joeddav/xlm-roberta-large-xnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Converts text into:input_ids, attention_mask
# max_length=106 is chosen based on job title length

def tokenize(batch):
  return tokenizer(batch["position"], truncation=True, padding = "max_length", max_length = 106)

train_dataset = train_dataset.map(tokenize, batched = True)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

eval_dataset = train_dataset.map(tokenize, batched = True)
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
train_dataset

### Model initialization for supervised classification

In [ ]:
 id2label = {int(k):str(v) for k,v in zip(codes, dpt)}
 label2id = {str(v):int(k) for k,v in id2label.items()}

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(label2id),
    id2label=id2label,
    label2id = label2id,
    ignore_mismatched_sizes = True
)

* We redefine the classification head to predict departments
* ignore_mismatched_sizes=True forces reinitialization of the output layer
* Consequence: The original NLI classification head (entailment / neutral / contradiction) is discarded

### Training and evaluation

In [ ]:
# Train the model
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(
    "test_trainer", report_to="none")

In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
  logits,labels = eval_pred
  predictions = np.argmax(logits, axis = -1)
  return metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = Trainer(
  model = model,
  args = training_args,
  train_dataset = train_dataset,
  eval_dataset = eval_dataset,
  compute_metrics = compute_metrics
)

trainer.evaluate()

## Result interpretation (after training)
## Accuracy = 13%

This is much worse than the 56% zero-shot accuracy.

### Why performance collapses after training

After traing, our model accuracy on evaluation data comes down to only 11%, compared to 56% before training. It seems that our model is broken after training.
Here are the possible reasons :

### Reason
xlm-roberta-large-xnli is not a standard classifier. It is trained for Natural Language Inference (NLI):
premise + hypothesis → {entailment, neutral, contradiction}
Zero-shot classification works because Hugging Face reformulates classification as NLI:
* Premise: The initial statement or context.
* Hypothesis: The statement being tested for truth relative to the premise.

Relationships:
1. Entailment: The premise logically implies the hypothesis (e.g., Premise: "A cat is on the mat." Hypothesis: "A feline is on a rug.").
2. Contradiction: The premise logically refutes the hypothesis (e.g., Premise: "The sun rises in the east." Hypothesis: "The sun never rises.").
3. Neutral: The hypothesis is neither confirmed nor denied by the premise.

When we fine-tuned it directly on:
*   input: job_position
*   label: department
The NLI alignment that makes zero-shot work is destroyed.


# Conclusion

* Zero-shot classification with xlm-roberta-large-xnli is: Effective
, Data-efficient, Suitable for this task.

* Fine-tuning this model directly is not appropriate.
It is not a standard classifier
Its strength lies in NLI-based zero-shot inference.

* For supervised learning, a better approach would be:
A model pretrained for sequence classification.
Or reformulating training data explicitly as NLI pairs

# Position-Department prediction with KNN

In [ ]:
import pandas as pd

In [ ]:
department = pd.read_csv("/content/DataScienceCapstoneProject/department.csv")
test_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")[["position","department"]]

## Embedding

We used the sentence transformer from TechWolf/JobBERT-v3 to embed our job positions for KNN.

* This is a sentence-transformers model specifically trained for job title matching and similarity.
* It's finetuned from sentence-transformers/paraphrase-multilingual-mpnet-base-v2 on a large dataset of job titles and their associated skills/requirements across multiple languages.

In [ ]:
! pip install sentence-transformers scikit-learn

In [ ]:
# SentenceTransformer expects a Python list of strings, not a pandas Series.

X = department["text"].astype(str).tolist()
y = department["label"].astype(str).tolist()

In [ ]:
# We split the "department.csv" dataset into the training data and evaluation data.
import numpy as np
from sklearn.model_selection import train_test_split

X = department["text"]
y = department['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state= 50, test_size = 0.2, shuffle = True
)

X_train = X_train.astype(str).tolist()
X_test = X_test.astype(str).tolist()
y_train = y_train.astype(str).tolist()
y_test = y_test.astype(str).tolist()

### Create Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("TechWolf/JobBERT-v3")
X_train_embedded = model.encode(
    X_train,
    show_progress_bar = True,
    normalize_embeddings = True
)
X_test_embedded = model.encode(
    X_test,
    show_progress_bar = True,
    normalize_embeddings = True
)

# Train KNN Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(
    n_neighbors = 5,
    metric = "cosine",
    weights="distance"
)

knn.fit(X_train_embedded, y_train)

# Evaluate

In [ ]:
for k in [3, 5, 7, 9, 11, 13, 15, 17, 19]:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(X_train_embedded, y_train)
    score = knn.score(X_test_embedded, y_test)
    print(f"K={k}, Accuracy={score:.3f}")

### we have the highest accuracy when k = 15 , so we will use k = 15 in the following evalution process. On the test dataset from the "department.csv", we got an accuracy of 93.3%

In [ ]:
k = 15
knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
knn.fit(X_train_embedded, y_train)

## Measure the accuracy on the CV dataset

In [ ]:
eval_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
X_eval = eval_df["position"].astype(str).tolist()
y_eval = eval_df["department"].astype(str).tolist()

X_eval_embedded = model.encode(
    X_eval,
    show_progress_bar = True,
    normalize_embeddings = True
)


In [ ]:
from sklearn.metrics import classification_report
y_pred = knn.predict(X_eval_embedded)
print(classification_report(y_eval, y_pred))

In the CV dataset, our prediction accuracy dropped to only 37%.
One of the main reasons is label disrtribution shift.
In our training data, the KNN model didn't get enough semantic training on the "Other" department, while in the test dataset half of the data is labeled as "Other".


## Measure the accuracy on the CV dataset without "Other" department

In [ ]:
eval_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
eval_df = eval_df[eval_df["department"]!= "Other"]
X_eval = eval_df["position"].astype(str).tolist()
y_eval = eval_df["department"].astype(str).tolist()

X_eval_embedded = model.encode(
    X_eval,
    show_progress_bar = True,
    normalize_embeddings = True
)

from sklearn.metrics import classification_report
y_pred = knn.predict(X_eval_embedded)
print(classification_report(y_eval, y_pred))


# KNN with distance threshold

In [ ]:
# Fit KNN normally
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(
    n_neighbors=15,
    metric="cosine",
    weights="distance"
)
knn.fit(X_train_embedded, y_train)

# Calculate the distances with the Linkedin data
eval_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
X_eval = eval_df["position"].astype(str).tolist()
y_eval = eval_df["department"].astype(str).tolist()

# Evaluate with the linkedinCV data
X_eval_embedded = model.encode(
    X_eval,
    show_progress_bar = True,
    normalize_embeddings = True
)

distances, indices = knn.kneighbors(X_eval_embedded)
# distances: shape (n_samples, k)

In [ ]:
from sklearn.metrics import accuracy_score
max_sim = 1- distances.min(axis = 1)
threshold = np.linspace(0.5,1.0,20)
# Calculate the smallest distance of our evalution sample to the nearest 15 training samples, to see if this evaluation sample is far from our training data.
# If the evaluation sample is too far from its nearest 15 training data neighbours,exceeding the threshold. We ditch it to "Other" department.
for t in threshold :
  is_other = max_sim < t
  y_pred = knn.predict(X_eval_embedded)
  y_pred_adj = y_pred.copy()
  y_pred_adj[is_other] = "Other"
  acc = accuracy_score(y_pred_adj, y_eval)
  print(f"Threshold ={t :.3f}, Accuracy={acc:.3f}")


# Accuracy = 63.3%
With our distance threshold method, if the sample is too far from our training data, we dump it to the "Other" label. It gives us a highest accuracy of 63.3%, with a similarity threshold of 0.816.

# Seniority prediction with KNN

In [ ]:
import pandas as pd

seniority = pd.read_csv("/content/DataScienceCapstoneProject/seniority.csv")
cv = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")

In [ ]:
trainCounts = seniority["label"].value_counts()
trainCounts["Professional"] = 0
testCounts = cv["seniority"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

counts = pd.DataFrame({
    "TrainingData" : trainCounts,
    "TestData" : testCounts
})

counts.plot(kind = "bar", figsize = (8,4))
plt.xlabel("Seniority")
plt.ylabel("Counts")
plt.title("Seniority Counts Comparison")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Here we also have the similar issue of label distribution drift like we have when we predict departments.
In the training data, we have around one third of "Lead" and one third of "Senior". But in the Test data, most of the data is labeled as "Professional", which doesn't appear in the Training data.

### We split the "seniority.csv" dataset into the training data and evaluation data.

In [ ]:

import numpy as np
from sklearn.model_selection import train_test_split

X = seniority["text"]
y = seniority['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state= 50, test_size = 0.2, shuffle = True
)

In [ ]:
# SentenceTransformer expects a Python list of strings, not a pandas Series.
X_train = X_train.astype(str).tolist()
y_train = y_train.astype(str).tolist()
X_test = X_test.astype(str).tolist()
y_test = y_test.astype(str).tolist()

## Create Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("TechWolf/JobBERT-v3")
X_train_embedded = model.encode(
    X_train,
    show_progress_bar = True,
    normalize_embeddings = True
)
X_test_embedded = model.encode(
    X_test,
    show_progress_bar = True,
    normalize_embeddings = True
)

## Train KNN classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
for k in [5, 7, 9, 11, 13, 15, 17 ,19, 21,23,25,27, 29,31]:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", weights = "distance")
    knn.fit(X_train_embedded, y_train)
    score = knn.score(X_test_embedded, y_test)
    print(f"K={k}, Accuracy={score:.3f}")

###  we have the highest accuracy when k = 21, so to prevent underfitting, we test a few different k values in the following evalution process. When we train on the test dataset in the "seniority.csv" dataset, we have a 79.5% accuracy rate.

In [ ]:
# Embed the positions in CV dataset
X_eval_embedded = model.encode(
    cv["position"].astype(str).tolist(),
    show_progress_bar = True,
    normalize_embeddings = True
)


## Test on CV dataset

In [ ]:
from sklearn.metrics import classification_report

knn = KNeighborsClassifier(n_neighbors=21, metric="cosine", weights = "distance")
knn.fit(X_train_embedded, y_train)
y_eval = cv["seniority"]
score = knn.score(X_eval_embedded, y_eval)
print(f"K=21, Accuracy={score:.3f}")

When k = 21, we have the highest accuracy rate on the linkedin CV seniority prediction, which is 33.7%

### KNN prediction on LInkedin CV Seniority without "Professional" labeled data

In [ ]:

cv = cv[cv["seniority"] != "Professional"]
X_eval_embedded = model.encode(
    cv["position"].astype(str).tolist(),
    show_progress_bar = True,
    normalize_embeddings = True
)

y_eval = cv["seniority"]
y_pred = knn.predict(X_eval_embedded)
print(classification_report(y_pred, y_eval))

Without the "Professional" labeled data, which is not included in the training data, the out-of-sample accuracy of our KNN model rises to 64%

# KNN-Seniority with distance threshold

In [ ]:
# Fit KNN normally
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(
    n_neighbors=21,
    metric="cosine",
    weights="distance"
)
knn.fit(X_train_embedded, y_train)

# Calculate the distances with the Linkedin data

y_eval = cv["seniority"]

X_eval_embedded = model.encode(
    cv["position"].astype(str).tolist(),
    show_progress_bar = True,
    normalize_embeddings = True
)

# Evaluate with the linkedinCV data
distances, indices = knn.kneighbors(X_eval_embedded)
# distances: shape (n_samples, k)

In [ ]:
from sklearn.metrics import accuracy_score
max_sim = 1- distances.min(axis = 1)
threshold = np.linspace(0.1,0.5,20)
# Calculate the smallest distance of our evalution sample to the nearest 21 training samples, to see if this evaluation sample is far from our training data.
# If the evaluation sample is too far from its nearest 21 training data neighbours,exceeding the threshold. We ditch it to Professional seniority.
for t in threshold :
  is_pro = max_sim < t
  y_pred = knn.predict(X_eval_embedded)
  y_pred_adj = y_pred.copy()
  y_pred_adj[is_pro] = "Professional"
  acc = accuracy_score(y_pred_adj, y_eval)
  print(f"Threshold ={t :.3f}, Accuracy={acc:.3f}")


With our distance threshold method, if the sample is too far from our training data, we dump it to the "Professional" label. It gives us a highest accuracy of 33.7%, which is the same as the accuracy before we apply this distance threshold methos. Distance threshold method doesn't help.

Possible reason :
Label ambiguity dominates the task.
LinkedIn titles like:
“Associate”
“Consultant”
“Staff”
“Member”
“Analyst”
are:
semantically very close,
but map to different seniority labels
So:
high cosine similarity ≠ clear label

### Evaluate the model without "Professional" labeled data

In [ ]:
cv = cv[cv["seniority"] != "Professional"]
X_eval_embedded = model.encode(
    cv["position"].astype(str).tolist(),
    show_progress_bar = True,
    normalize_embeddings = True
)

y_eval = cv["seniority"]
y_pred = knn.predict(X_eval_embedded)
y_pred_adj = y_pred.copy()
acc = accuracy_score(y_pred_adj, y_eval)
coverage = 1 - reject.mean()
print(f"t={t:.3f}, acc={acc:.3f}, coverage={coverage:.2%}")